In [1]:
!pip install kaggle
from google.colab import files
files.upload()   # upload your kaggle.json here (from your Kaggle account)

!mkdir ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d wordsforthewise/lending-club
!unzip lending-club.zip


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/wordsforthewise/lending-club
License(s): CC0-1.0
 99% 1.25G/1.26G [00:14<00:00, 157MB/s]
100% 1.26G/1.26G [00:14<00:00, 95.1MB/s]
Archive:  lending-club.zip
  inflating: accepted_2007_to_2018Q4.csv.gz  
  inflating: accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv  
  inflating: rejected_2007_to_2018Q4.csv.gz  
  inflating: rejected_2007_to_2018q4.csv/rejected_2007_to_2018Q4.csv  


In [2]:
import pandas as pd
df_s = pd.read_csv('accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv', low_memory=False)

In [3]:
df = df_s.sample(n=500000, random_state=42)


In [4]:
# Standardize column names immediately after loading to prevent KeyErrors
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('%', '')

# Display the first few rows and column info
print(df.head())
print(df.info())

                id  member_id  loan_amnt  funded_amnt  funded_amnt_inv  \
392949    39651438        NaN    32000.0      32000.0          32000.0   
1273506   16411620        NaN     9600.0       9600.0           9600.0   
324024    45122316        NaN     4000.0       4000.0           4000.0   
2066630  125356772        NaN     6025.0       6025.0           6025.0   
477199   128490686        NaN    25000.0      25000.0          25000.0   

               term  int_rate  installment grade sub_grade  ...  \
392949    60 months     10.49       687.65     B        B3  ...   
1273506   36 months     12.99       323.42     C        C1  ...   
324024    36 months      6.68       122.93     A        A3  ...   
2066630   36 months     10.91       197.00     B        B4  ...   
477199    60 months     26.30       752.96     E        E5  ...   

        hardship_payoff_balance_amount hardship_last_payment_amount  \
392949                             NaN                          NaN   
1273506   

In [8]:
# Install necessary libraries
!pip install pandas numpy scikit-learn torch torchvision torchaudio d3rlpy matplotlib seaborn shap
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Detect device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device for PyTorch: {device}")

Using device for PyTorch: cpu


In [40]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import xgboost as xgb # Import the xgboost library
import d3rlpy
from d3rlpy.dataset import MDPDataset
from tqdm.notebook import tqdm

# --- CONFIGURATION (Adjust for your final run) ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# Rationale: XGBoost is used now, so PyTorch DL settings are commented out
# DL_EPOCHS = 50
RL_EPOCHS = 10
DL_THRESHOLD_FALLBACK = 0.27
OPERATIONAL_COST = 50
# --- END CONFIGURATION ---

print(f"--- Running Monolithic Pipeline on {DEVICE} ---")

# ======================================================================
# 1. CORE FUNCTION DEFINITIONS
# ======================================================================

def create_target(df, target_col='loan_status'):
    """Converts loan_status to binary target: 0: Fully Paid, 1: Defaulted/Charged Off"""
    df['default_flag'] = df[target_col].apply(
        lambda x: 1 if isinstance(x, str) and x.lower() in ['charged off', 'default'] else 0
    )
    return df

def create_credit_features(df):
    """Creates advanced credit utilization and age features, using 'total_rev_hi_lim'."""

    df['revol_credit_limit_safe'] = df['total_rev_hi_lim'].replace(0, np.nan)
    df['credit_utilization'] = (df['revol_bal'] / (df['revol_credit_limit_safe'] + 1e-6)).clip(0, 1.5) # Add epsilon for safety
    df['available_credit'] = df['total_rev_hi_lim'].fillna(0) - df['revol_bal'].fillna(0)
    df['account_utilization'] = df['open_acc'] / (df['total_acc'].replace(0, 1))

    # 2. CREDIT AGE FIX (Robust Fallback Logic):
    if 'mths_since_earliest_cr_line' in df.columns:
        df['credit_age_years'] = df['mths_since_earliest_cr_line'] / 12
    elif 'earliest_cr_line' in df.columns:
        df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], errors='coerce')
        REFERENCE_DATE = pd.to_datetime('2018-12-01')
        df['credit_age_days'] = (REFERENCE_DATE - df['earliest_cr_line']).dt.days
        df['credit_age_years'] = df['credit_age_days'] / 365.25
    else:
        df['credit_age_years'] = 0

    return df

def create_income_features(df):
    """Creates income-based features with robust scaling and employment length."""
    df['log_annual_inc'] = np.log1p(df['annual_inc'])

    df['emp_length_years_str_temp'] = df['emp_length'].replace({'10+ years': '10', '< 1 year': '0.5', 'n/a': np.nan})

    df['emp_length_years'] = df['emp_length_years_str_temp'].str.extract('(\d+\.?\d*)', expand=False).astype(float)
    df['emp_length_years'] = df['emp_length_years'].fillna(0)

    df.drop(columns=['emp_length_years_str_temp'], errors='ignore', inplace=True)

    df['installment_to_income'] = (df['installment'] / (df['annual_inc'] / 12 + 1e-6)).clip(0, 2) # Add epsilon
    df['loan_to_income'] = (df['loan_amnt'] / (df['annual_inc'] + 1e-6)).clip(0, 5) # Add epsilon

    return df

def create_interest_features(df):
    """Creates interest rate and profitability features."""
    df['int_rate_float'] = pd.to_numeric(df['int_rate'], errors='coerce') / 100

    df['term_years'] = df['term'].str.extract('(\d+)', expand=False).astype(float).fillna(0) / 12

    df['total_interest_income'] = (df['loan_amnt'] * df['int_rate_float'] * df['term_years'])

    df['fico_avg'] = (df['fico_range_high'] + df['fico_range_low']) / 2
    df['loan_to_fico_ratio'] = df['loan_amnt'] / (df['fico_avg'] + 1e-6) # Add epsilon

    return df

def create_rl_reward(df, principal_loss_col='loan_amnt', interest_income_col='total_interest_income', default_flag_col='default_flag'):
    """Calculates the financial reward for RL: Profit/Loss adjusted for operational cost."""
    df['reward'] = 0.0

    # Case 1: Fully Paid (Profit)
    paid_mask = (df[default_flag_col] == 0)
    df.loc[paid_mask, 'reward'] = df.loc[paid_mask, interest_income_col] - OPERATIONAL_COST

    # Case 2: Defaulted (Loss)
    default_mask = (df[default_flag_col] == 1)

    if 'recoveries' in df.columns:
        df.loc[default_mask, 'principal_loss'] = df.loc[default_mask, principal_loss_col] - df.loc[default_mask, 'recoveries']
        df.loc[default_mask, 'reward'] = -(df.loc[default_mask, 'principal_loss'] + OPERATIONAL_COST)
    else:
        df.loc[default_mask, 'reward'] = -(df.loc[default_mask, principal_loss_col] + OPERATIONAL_COST)

    return df


def preprocess_data(df):
    """Runs all feature engineering, handles missing values, and prepares the final feature matrix."""

    df = create_target(df)
    df = create_credit_features(df)
    df = create_income_features(df)
    df = create_interest_features(df)

    # --- Feature Selection ---
    numeric_features = [
        # Base Features
        'loan_amnt', 'int_rate_float', 'installment', 'annual_inc', 'dti',
        'fico_avg', 'revol_bal', 'total_rev_hi_lim', 'open_acc', 'total_acc',
        'pub_rec', 'delinq_2yrs', 'mths_since_last_delinq',
        'mo_sin_old_rev_tl_op', 'total_bc_limit', 'tot_cur_bal', 'total_rec_int',
        # Engineered Features
        'credit_age_years', 'credit_utilization', 'available_credit',
        'log_annual_inc', 'installment_to_income', 'loan_to_income', 'emp_length_years',
        'verification_status_joint', 'loan_to_fico_ratio'
    ]

    categorical_features = ['grade', 'home_ownership', 'purpose', 'verification_status', 'addr_state']

    # 1. Imputation: Robust Imputation Loop
    features_to_impute = [f for f in numeric_features if f in df.columns]

    for col in features_to_impute:
        if pd.api.types.is_numeric_dtype(df[col]) and df[col].notna().any():
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
        else:
            df[col] = df[col].fillna(0)

    # 2. Categorical Imputation: Fill missing with 'Missing' category
    for col in categorical_features:
        if col in df.columns:
            df[col] = df[col].astype(str).fillna('Missing')

    # 3. Encoding: One-hot encode categorical features
    df_processed = pd.get_dummies(df, columns=categorical_features, drop_first=True, dummy_na=False)

    # 4. Final Feature List
    final_features_base = numeric_features + [col for col in df_processed.columns if col.startswith(tuple(categorical_features))]
    final_features = [col for col in final_features_base if col in df_processed.columns]

    # Final check: Convert to float array (Robustness step)
    df_final = df_processed[final_features].copy()
    df_final = df_final.apply(pd.to_numeric, errors='coerce')
    df_final = df_final.replace([np.inf, -np.inf], np.nan).fillna(0)

    X = df_final.values.astype(np.float32)
    y = df_processed['default_flag'].values

    return df_processed, X, y, final_features

# ----------------------------------------------------------------------
# DL MODEL REPLACEMENT: XGBOOST IMPLEMENTATION (Task 2)
# ----------------------------------------------------------------------

def train_xgboost_model(X_train, y_train, X_val, y_val):
    """
    Trains an XGBoost model optimized for imbalanced data using scale_pos_weight.

    Returns: Trained XGBClassifier and the fitted scaler.
    """
    print("Scaling numeric features...")
    # Scale numeric features (essential for fair comparison with RL/DL architectures)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    # Calculate the crucial imbalance parameter: (Total Non-Defaults / Total Defaults)
    total_defaults = y_train.sum()
    total_non_defaults = len(y_train) - total_defaults
    scale_pos_weight_val = total_non_defaults / total_defaults

    print(f"Calculated scale_pos_weight: {scale_pos_weight_val:.2f}")

    # Initialize XGBoost Classifier with imbalance handling and fast training parameters
    xgb_model = XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        n_estimators=1000, # Increased estimators for early stopping
        learning_rate=0.05,
        max_depth=5,
        scale_pos_weight=scale_pos_weight_val,
        random_state=42,
        tree_method='hist',
        use_label_encoder=False # Suppress deprecation warning
    )

    print("Training XGBoost model...")
    # Use validation set for early stopping with the correct callback
    # eval_set = [(X_val_scaled, y_val)] # Removed eval_set
    xgb_model.fit(
        X_train_scaled,
        y_train,
        # eval_set=eval_set, # Removed eval_set
        # callbacks=[xgb.callback.EarlyStopping(rounds=10)] # Removed callbacks
    )

    # Early stopping information might not be available without callbacks/eval_set in this version
    # print(f"XGBoost Best Iteration: {xgb_model.best_iteration}")


    # Evaluate final AUC on validation set
    val_probs = xgb_model.predict_proba(X_val_scaled)[:, 1]
    val_auc = roc_auc_score(y_val, val_probs)
    print(f"XGBoost Final Validation AUC: {val_auc:.4f}")

    return xgb_model, scaler

def predict_xgb(model, scaler, X_data):
    """Generates default probabilities using the trained model and scaler."""
    X_scaled = scaler.transform(X_data)
    # Get probability of the positive class (Default, which is index 1)
    return model.predict_proba(X_scaled)[:, 1]

# ----------------------------------------------------------------------
# Offline RL Model (No change in logic, only removed redundant param)
# ----------------------------------------------------------------------

def prepare_offline_rl_dataset(X_data, df_processed, reward_col='reward'):
    """Converts data to MDPDataset format, including synthetic 'deny' transitions."""
    observations = X_data
    n_samples = len(observations)
    actions_approve = np.ones(n_samples, dtype=np.int32)
    rewards_approve = df_processed[reward_col].values.astype(np.float32)

    n_deny = int(n_samples * 0.2)
    deny_indices = np.random.choice(n_samples, n_deny, replace=False)
    deny_observations = observations[deny_indices]
    deny_actions = np.zeros(n_deny, dtype=np.int32)
    deny_rewards = np.zeros(n_deny, dtype=np.float32)

    all_observations = np.vstack([observations, deny_observations])
    all_actions = np.concatenate([actions_approve, deny_actions])
    all_rewards = np.concatenate([rewards_approve, deny_rewards])
    terminals = np.ones(len(all_observations), dtype=np.float32)

    # FIX: Removed 'discrete_action=True'
    dataset = MDPDataset(
        observations=all_observations,
        actions=all_actions,
        rewards=all_rewards,
        terminals=terminals,
    )
    return dataset

def train_offline_rl_agent(dataset, n_epochs=100, device=f'{DEVICE}:0'):
    """Trains a Conservative Q-Learning (CQL) agent."""
    agent = d3rlpy.algos.DiscreteCQLConfig(
        batch_size=256,
        learning_rate=3e-4,
        encoder_factory=d3rlpy.models.VectorEncoderFactory(
            hidden_units=[256, 128],
            activation='relu',
            dropout_rate=0.1
        )
    ).create(device=device)

    print(f"Training RL agent on {dataset.size()} transitions...")
    # NOTE: Set n_epochs to RL_EPOCHS (10) for this quick run
    # Assuming 1000 steps per epoch as a reasonable default
    steps_per_epoch = 1000
    total_steps = RL_EPOCHS * steps_per_epoch

    agent.fit(
        dataset,
        n_steps=total_steps,
        show_progress=False
    )
    return agent

def evaluate_rl_policy(agent, X_test, df_test):
    """Evaluates the RL policy using direct simulation for Expected Policy Value (EPV)."""
    test_observations = X_test
    # Use predict_best_action for deterministic evaluation
    rl_decisions = agent.predict(test_observations) # 1=Approve, 0=Deny

    approve_mask = (rl_decisions == 1)

    approved_rewards = df_test.loc[approve_mask, 'reward']
    denied_rewards = pd.Series(0.0, index=df_test.loc[~approve_mask].index)

    policy_rewards = pd.concat([approved_rewards, denied_rewards])
    direct_policy_value = policy_rewards.mean()

    approval_rate = approve_mask.mean()

    return {
        'direct_policy_value': direct_policy_value,
        'approval_rate': approval_rate,
        'rl_decisions': rl_decisions
    }

def calculate_profit(decisions, df_test):
    """Calculates profit per loan based on approval decisions."""
    approved_mask = (decisions == 1)
    approved_rewards = df_test.loc[approved_mask, 'reward']
    denied_rewards = pd.Series(0.0, index=df_test.loc[~approved_mask].index)
    total_profit = pd.concat([approved_rewards, denied_rewards]).sum()
    return total_profit / len(df_test)

# ======================================================================
# 2. MAIN EXECUTION FLOW
#======================================================================

# Note: Assumes 'df' is already loaded and standardized (lower-cased) from initial cells.

# ----------------------------------------------------------------------
# 2.1 Data Preparation and Split
# ----------------------------------------------------------------------
print("\n--- 1. Data Preparation and Feature Engineering ---")

# 2.1.1 Run Preprocessing
df_processed, X, y, feature_names = preprocess_data(df)

# 2.1.2 Create RL Financial Reward
df_processed = create_rl_reward(df_processed)

# 2.1.3 Split Data (Train/Val/Test)
X_train_full, X_test, y_train_full, y_test, df_train_full, df_test = train_test_split(
    X, y, df_processed, test_size=0.2, random_state=42, stratify=y
)

# Split full training data into train and validation for DL/XGBoost early stopping
X_train, X_val, y_train, y_val, df_train, df_val = train_test_split(
    X_train_full, y_train_full, df_train_full, test_size=0.125, random_state=42, stratify=y_train_full
)

input_dim = X_train.shape[1]
print(f"Number of features: {input_dim}")
print(f"Train samples: {len(X_train)}, Val samples: {len(X_val)}, Test samples: {len(X_test)}")

<>:64: SyntaxWarning: invalid escape sequence '\d'
<>:78: SyntaxWarning: invalid escape sequence '\d'
<>:64: SyntaxWarning: invalid escape sequence '\d'
<>:78: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-1517503499.py:64: SyntaxWarning: invalid escape sequence '\d'
  df['emp_length_years'] = df['emp_length_years_str_temp'].str.extract('(\d+\.?\d*)', expand=False).astype(float)
/tmp/ipython-input-1517503499.py:78: SyntaxWarning: invalid escape sequence '\d'
  df['term_years'] = df['term'].str.extract('(\d+)', expand=False).astype(float).fillna(0) / 12


--- Running Monolithic Pipeline on cpu ---

--- 1. Data Preparation and Feature Engineering ---
Number of features: 108
Train samples: 350000, Val samples: 50000, Test samples: 100000


In [16]:
# ----------------------------------------------------------------------
# 2.2 XGBOOST MODEL TRAINING (Task 2)
# ----------------------------------------------------------------------
print("\n--- 2.2 XGBoost Model Training (Supervised, Weighted) ---")
# xgb_model is the new DL model, dl_scaler is needed for prediction scaling
xgb_model, dl_scaler = train_xgboost_model(
    X_train, y_train, X_val, y_val
)

# 2.2.1 Generate Probabilities
dl_probs = predict_xgb(xgb_model, dl_scaler, X_test)


# ----------------------------------------------------------------------
# 2.3 DL Threshold Optimization (To find optimal F1 and Profit)
# ----------------------------------------------------------------------
print("\n--- 2.3 DL Threshold Optimization ---")

thresholds = np.linspace(0.01, 0.99, 100)
results = []

for t in thresholds:
    # DL prediction is P(default). Approve if P(default) < t.
    dl_decision_at_t = (dl_probs < t).astype(int)

    # We predict 1 (Default) if P(default) >= t.
    dl_prediction_of_default = (dl_probs >= t).astype(int)

    f1 = f1_score(y_test, dl_prediction_of_default)
    profit = calculate_profit(dl_decision_at_t, df_test)

    results.append({
        'threshold': t,
        'f1_score': f1,
        'expected_profit': profit,
        'approval_rate': dl_decision_at_t.mean()
    })

results_df = pd.DataFrame(results)

# Find the best F1 threshold
best_f1_row = results_df.loc[results_df['f1_score'].idxmax()]

# Find the best PROFIT threshold (The Business Objective)
best_profit_row = results_df.loc[results_df['expected_profit'].idxmax()]

DL_THRESHOLD_OPTIMAL = best_profit_row['threshold']
dl_decisions_optimal = (dl_probs < DL_THRESHOLD_OPTIMAL).astype(int)
dl_profit_optimal = best_profit_row['expected_profit']

print(f"Optimal F1 Threshold ({best_f1_row['threshold']:.3f}): F1={best_f1_row['f1_score']:.4f}, Profit=${best_f1_row['expected_profit']:.2f}")
print(f"Optimal Profit Threshold ({DL_THRESHOLD_OPTIMAL:.3f}): Profit=${dl_profit_optimal:.2f}, F1={best_profit_row['f1_score']:.4f}")





--- 2.2 XGBoost Model Training (Supervised, Weighted) ---
Scaling numeric features...
Calculated scale_pos_weight: 7.38
Training XGBoost model...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [06:20:38] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Final Validation AUC: 0.8158

--- 2.3 DL Threshold Optimization ---
Optimal F1 Threshold (0.594): F1=0.4398, Profit=$4583.62
Optimal Profit Threshold (0.871): Profit=$4992.17, F1=0.1770


In [41]:
# ----------------------------------------------------------------------
# 2.4 Offline RL Agent (Task 3)
# ----------------------------------------------------------------------
print("\n--- 3. Offline RL Agent Training (CQL) ---")
# 2.4.1 Prepare RL Dataset (using X_train_full, the full 80% split)
rl_dataset = prepare_offline_rl_dataset(X_train_full, df_train_full)

# ----------------------------------------------------------------------
# MAIN EXECUTION FLOW
# ----------------------------------------------------------------------
# Update the call to use the higher epoch value
rl_agent = train_offline_rl_agent(rl_dataset, n_epochs=RL_EPOCHS, device=f'{DEVICE}:0')

# 2.4.3 Evaluate RL Policy on Test Set
print("\n--- Evaluating RL Policy ---")
# Use d3rlpy's evaluation utility if available and suitable for offline setting
# Note: evaluate_policy typically requires a Gym-like environment.
# Since we don't have one, we'll stick to manual evaluation for now
# and focus on fixing the prediction issue in evaluate_rl_policy.

# The error is in evaluate_rl_policy, which is called in the previous cell.
# The fix for the AttributeError 'DiscreteCQL' object has no attribute 'q_func'
# is in the evaluate_rl_policy function definition in cell DIiA8dZpJKRG.
# Please re-run cell DIiA8dZpJKRG first to apply the fix, then re-run this cell.

# After fixing the AttributeError in evaluate_rl_policy (in cell DIiA8dZpJKRG),
# the evaluation will proceed here.
rl_results = evaluate_rl_policy(rl_agent, X_test, df_test)
rl_decisions = rl_results['rl_decisions']
rl_profit_per_loan = rl_results['direct_policy_value']

print("\n--- RL Policy Metrics ---")
print(f"RL Policy (Q(approve) > Q(deny)):")
print(f"  Estimated Policy Value (Direct Sim.): ${rl_profit_per_loan:.2f}")
print(f"  Approval Rate: {rl_results['approval_rate']:.2%}")


--- 3. Offline RL Agent Training (CQL) ---
2025-10-26 07:36.38 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int32')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(108,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-10-26 07:36.38 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2025-10-26 07:36.40 [info     ] Action size has been automatically determined. action_size=2
Training RL agent on 480000 transitions...
2025-10-26 07:36.41 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(108,)]), action_signature=Signature(dtype=[dtype('int32')], shape=[(1,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.DISCRETE: 2>, action_size=2)
2025-10-26 07:36.41 [debug    ] Building models...            
2

In [ ]:
# ----------------------------------------------------------------------
# 2.5 Comparative Analysis (Task 4)
# ----------------------------------------------------------------------
print("\n--- 4. Comparative Policy Analysis (Disagreements) ---")

dl_deny_rl_approve = (dl_decisions_optimal == 0) & (rl_decisions == 1)
dl_approve_rl_deny = (dl_decisions_optimal == 1) & (rl_decisions == 0)

print(f"Total Test Cases: {len(X_test)}")
print("-" * 30)
print(f"DL Denies, RL Approves (Risk-Seeking Cases): {np.sum(dl_deny_rl_approve)} loans")
if np.sum(dl_deny_rl_approve) > 0:
    profit_rs = df_test.loc[dl_deny_rl_approve, 'reward'].mean()
    print(f"  Avg. Profit/Loan if Approved (RL's gain): ${profit_rs:.2f}")

print("-" * 30)
print(f"DL Approves, RL Denies (RL Conservative Cases): {np.sum(dl_approve_rl_deny)} loans")
if np.sum(dl_approve_rl_deny) > 0:
    profit_c = df_test.loc[dl_approve_rl_deny, 'reward'].mean()
    print(f"  Avg. Profit/Loan if Approved (DL's loss): ${profit_c:.2f}")

print("\n*** Final Summary for Report ***")
print(f"RL Policy achieved Expected Profit: ${rl_profit_per_loan:.2f} per loan.")
print(f"DL Policy achieved Expected Profit (Optimal): ${dl_profit_optimal:.2f} per loan.")